In [7]:
import pandas as pd
import numpy as np

In [9]:
df = pd.read_csv("../data/processed/fut_prices_clean.csv", parse_dates= ["date"])
df.head()

,player_id,player_name,date,price,pct_change
0,39,lionel-messi,2023-09-21,661957.0,NaN
1,39,lionel-messi,2023-09-22,159944.0,-0.758377
2,39,lionel-messi,2023-09-23,172778.0,0.080241
3,39,lionel-messi,2023-09-24,215429.0,0.246854
4,39,lionel-messi,2023-09-25,262156.0,0.216902


In [10]:
#create lag features
df["lag_1"] = df.groupby("player_id")["price"].shift(1)
df["lag_7"] = df.groupby("player_id")["price"].shift(7)
df["lag_30"] = df.groupby("player_id")["price"].shift(30)

In [11]:
#create moving average features
df["ma_7"] = df.groupby("player_id")["price"].rolling(7).mean().reset_index(0, drop= True)
df["ma_14"] = df.groupby("player_id")["price"].rolling(14).mean().reset_index(0, drop= True)
df["ma_30"] = df.groupby("player_id")["price"].rolling(30).mean().reset_index(0, drop= True)

In [12]:
#create volatility features
df["volatility_7"] = df.groupby("player_id")["price"].rolling(7).std().reset_index(0, drop= True)
df["volatility_30"] = df.groupby("player_id")["price"].rolling(30).std().reset_index(0, drop= True)

In [13]:
#create momentum features
df["momentum_7"] = df["price"] - df["lag_7"]
df["momentum_30"] = df["price"] - df["lag_30"]

These are the following reason why we created these four variables:

- Lag Features: Previous values shifted back in time (like yesterday's price). They capture direct historical relationships since today's price often correlates with recent prices.

- Moving Average: The average price over a rolling window that smooths out short-term fluctuations. It reveals the underlying trend and identifies support/resistance levels.

- Volatility: A statistical measure of price dispersion (how much prices fluctuate). It indicates risk, uncertainty, and market activity levels.

- Momentum: The rate of price change over a specific period. It shows trend strength, direction, and whether price movement is accelerating or slowing down.

Together, they will give the model a complete picture of price dynamics of the fut economy.

In [14]:
#create day of the week, month, and weekend feature
df["day_of_week"] = df["date"].dt.day_of_week
df["month"] = df["date"].dt.month
df["weekend"] = np.where(df["day_of_week"] >= 5, 1, 0)

In [24]:
#make player_id as a category instead of an integer variable
df["player_id"] = df["player_id"].astype("category")
df["player_id"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 3982 entries, 0 to 3981
Series name: player_id
Non-Null Count  Dtype   
--------------  -----   
3982 non-null   category
dtypes: category(1)
memory usage: 4.4 KB


In [25]:
df.head()

,player_id,player_name,date,price,pct_change,lag_1,lag_7,lag_30,ma_7,ma_14,ma_30,volatility_7,volatility_30,momentum_7,momentum_30,day_of_week,month,weekend
0,39,lionel-messi,2023-09-21,661957.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,9,0
1,39,lionel-messi,2023-09-22,159944.0,-0.758377,661957.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,9,0
2,39,lionel-messi,2023-09-23,172778.0,0.080241,159944.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,9,1
3,39,lionel-messi,2023-09-24,215429.0,0.246854,172778.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,9,1
4,39,lionel-messi,2023-09-25,262156.0,0.216902,215429.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,9,0


In [26]:
#save the data with the new features in a csv
df.to_csv("../data/processed/fut_prices_features.csv", index= False)